In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_email AS (
WITH deduped_id_activities_filtered_email AS (
    SELECT
        a.id AS acitvity_id,
        a.lead_id, 
        a.type,
        a.status,
        a.activity_at,
        b.body_text,
        ROW_NUMBER() OVER (PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed a
    LEFT JOIN
        crm_ingestion.silver.lead_activites_email_body b
    ON
        a.id = b.activity_id
    WHERE 
        a.type = 'Email'
    AND a.status = 'inbox'
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%A new event has been scheduled.%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%is requesting access to the following folder%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%requests access to an item%')
),
latest_email_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_email
    WHERE
        id_rn = 1
)
    SELECT
        lead_id,
        DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_email
    FROM
        latest_email_per_lead_id
    WHERE
        lead_id_rn = 1
)